## 8.1 Regular 1D grids

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import xarray as xr
import scipy.interpolate
from matplotlib import pyplot as plt
import matplotlib.patches as patches
from cartopy import crs as ccrs
import cartopy.feature as cfeature
from pyresample import geometry
from pyresample.kd_tree import resample_nearest

In [ ]:
plt.rcParams.update({"font.size": 16, 
    "figure.figsize": [12, 6],
    "font.sans-serif": ["Segoe UI", "DejaVu Sans", "sans-serif"]
    })

In [ ]:
filename = "hms_fire20250109.txt"
fires_path = Path("./data/txt") / filename
fires = pd.read_csv(fires_path, skipinitialspace=True)
fires.head()

In [ ]:
extent_global = [-180.0 , -90.0 , 180.0 , 90.0]

grid_size = 1.0

num_points_x = int((extent_global[2] - extent_global[0])/grid_size)
num_points_y = int((extent_global[3] - extent_global[1])/grid_size)

print(num_points_x, num_points_y)

In [ ]:
# Using a real step length will skip the end value
print(np.mgrid[0:4:1.0])

# Using a complex number of bins will include the end value
print(np.mgrid[0:4:5j])

In [ ]:
fires_nx = complex(0, num_points_x)
fires_ny = complex(0, num_points_y)

fires_Xnew, fires_Ynew = np.mgrid[extent_global[0]:extent_global[2]:nx, \
                      extent_global[1]:extent_global[3]:ny]

In [ ]:
fires_count = np.zeros([num_points_x, num_points_y])

for i, lon in enumerate(fires.Lon):
    lat = fires.Lat[i]
    
    adjlat = ((lat + 90) / grid_size)
    adjlon = ((lon + 180 ) / grid_size)
    
    latbin = int(adjlat)
    lonbin = int(adjlon)
    
    fires_count[lonbin, latbin] = fires_count[lonbin, latbin]+1

In [ ]:
fires_count

In [ ]:
fires_count[fires_count == 0] = np.nan

In [ ]:
extent_na = [-170, -50, 14, 72]

fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=[12, 6], \
           constrained_layout=True, subplot_kw={"projection": ccrs.PlateCarree()})

ax1.set_title("Scatter on Irregular Grid")
ax1.coastlines('10m')
ax1.set_extent(extent_na)
left = ax1.scatter(fires["Lon"], fires["Lat"], s=1)

ax2.set_title("Gridded and Counted to 1°×1°")
ax2.coastlines('10m')
ax2.set_extent(extent_na)
right = ax2.pcolormesh(fires_Xnew, fires_Ynew, fires_count, vmin=0, vmax=200)

fig.colorbar(right, orientation="vertical", ax=ax2, pad=0.1, 
             label="Counts of Fires", shrink=0.5)
plt.show()

## 8.2 Regular 2D grids

In [ ]:
filename = "3B-HHR.MS.MRG.3IMERG.20240927-S073000-E075959.0450.V07B.HDF5"
imerg_path = Path("./data/imerg") / filename

In [ ]:
imerg = xr.open_datatree(imerg_path, engine="h5netcdf", decode_timedelta=True)

imerg_lat = imerg["Grid/lat"]
imerg_lon = imerg["Grid/lon"]
imerg_precip = imerg["Grid/precipitation"].isel(time=0)

In [ ]:
imerg_Xold, imerg_Yold = np.meshgrid(imerg_lon, imerg_lat, indexing="ij")

In [ ]:
# Number of nx and ny points for the grid. 720 nx, 360 ny creates 1.0 degree grid.
def create_2d_grid(grid_size):
    coverage = [-180.0 , -90.0 , 180.0 , 90.0]

    num_points_x = int((coverage[2] - coverage[0])/grid_size)
    num_points_y = int((coverage[3] - coverage[1])/grid_size)

    nx = complex(0, num_points_x)
    ny = complex(0, num_points_y)

    Xnew, Ynew = np.mgrid[coverage[0]:coverage[2]:nx, coverage[1]:coverage[3]:ny]
    return Xnew, Ynew

In [ ]:
imerg_Xnew, imerg_Ynew = create_2d_grid(0.5)

In [ ]:
imerg_precip.shape, imerg_Yold.shape, imerg_Ynew.shape

In [ ]:
np.array(imerg_precip, copy=True).flatten()

In [ ]:
imerg_values = np.array(imerg_precip, copy=True).flatten()

dims = (imerg_values.shape[0], 2)
imerg_points =  np.zeros(dims)

imerg_points[:, 0] = imerg_Xold.flatten()
imerg_points[:, 1] = imerg_Yold.flatten()

In [ ]:
imerg_grid_out = scipy.interpolate.griddata(imerg_points, imerg_values, (imerg_Xnew, imerg_Ynew), method="nearest")

In [ ]:
extent_se_usa = [-90, -75, 40, 30]
imerg_label="Rainrate (mm/hr)"

fig, axes = plt.subplots(ncols=2, figsize=[12, 6], \
   subplot_kw={"projection": ccrs.PlateCarree()})

for ax in axes:
    ax.set_extent(extent_se_usa)
    ax.add_feature(cfeature.STATES, edgecolor="white")

axes[0].set_title("Native Grid (0.1°×0.1°)")
left = axes[0].pcolormesh(imerg_Xold, imerg_Yold, imerg_precip, vmin=0, vmax=20)

axes[1].set_title("Coarsened Grid (1°×1°)")
right = axes[1].pcolormesh(imerg_Xnew, imerg_Ynew, imerg_grid_out, vmin=0, vmax=20)

fig.colorbar(right, orientation="horizontal", ax=axes, pad=0.1, 
             label=imerg_label, shrink=0.8)

plt.show()

## 8.3 Irregular 2D grids
### 8.3.1 Resizing

In [ ]:
filename = "OR_ABI-L1b-RadM2-M6C02_G19_s20252981300559_e20252981301017_c20252981301044.nc"
abi_path = Path("./data/abi") / filename
abi = xr.open_dataset(abi_path, engine="h5netcdf")

In [ ]:
abi_rad = abi["Rad"]
abi_x = abi["x"]
abi_y = abi["y"]

In [ ]:
def get_sat_params(fid):
  geom = fid.goes_imager_projection

  # In meters and degrees
  satellite_params = {
  "central_lon": geom.longitude_of_projection_origin,
  "central_lat": 0.0,
  "sat_height": geom.perspective_point_height,
  "semi_major_axis" : geom.semi_major_axis,
  "semi_minor_axis" : geom.semi_minor_axis,
  "R_earth" : 6371000
  }

  return satellite_params

In [ ]:
abi_params = get_sat_params(abi)
globe = ccrs.Globe(semimajor_axis=abi_params["semi_major_axis"], semiminor_axis=abi_params["semi_minor_axis"])
geo_crs = ccrs.Geostationary(central_longitude=abi_params["central_lon"], satellite_height=abi_params["sat_height"], globe=globe)

X = abi.variables["x"] * abi_params["sat_height"]
Y = abi.variables["y"] * abi_params["sat_height"]

abi_extent = (X.min(), X.max(), Y.min(), Y.max())

In [ ]:
abi_rad_resized = abi_rad[::20, ::20]

In [ ]:
abi_lims = [0, 300]
abi_label = "Brightness Temperature (K)"

fig, axes = plt.subplots(ncols=2, figsize=[12, 6], \
     subplot_kw={"projection": geo_crs})

for ax in axes:
    ax.set_extent(abi_extent, crs=geo_crs)
    ax.coastlines("10m", color="black")

axes[0].set_title("Native Resolution")
left = axes[0].imshow(abi_rad, cmap="rainbow", origin="upper", 
            extent=abi_extent, vmin=abi_lims[0], vmax=abi_lims[1], transform=geo_crs)

axes[1].set_title("20x Lower Resolution")
right = axes[1].imshow(abi_rad_resized, cmap="rainbow", origin="upper", 
            extent=abi_extent, vmin=abi_lims[0], vmax=abi_lims[1])

fig.colorbar(right, orientation="horizontal", ax=axes, pad=0.1, 
             label=abi_label, shrink=0.8)

plt.show()

### 8.3.2 Regridding

In [ ]:
patch_jm = (-76.75, 17.25)
delta_jm = 1
extent_jm = (-76.75, -75.75, 17.25, 18.25)

fig, axes = plt.subplots(ncols=1, nrows=1, figsize=[6, 6], \
     subplot_kw={"projection": geo_crs})

axes.set_extent(abi_extent, crs=geo_crs)
# axes.set_extent(extent_jm, crs=ccrs.PlateCarree())

axes.coastlines("10m", color="black")

meso = axes.imshow(abi_rad, cmap="tab20c_r", origin="upper", \
     extent=abi_extent, vmin=abi_lims[0], vmax=abi_lims[1], \
     transform=geo_crs)

fig.colorbar(meso, orientation="horizontal", pad=0.05, label=abi_label)

zoom_box = patches.Rectangle(patch_jm, delta_jm, delta_jm, 
     linewidth=4, edgecolor="black", facecolor="none", \
     transform=ccrs.PlateCarree())

axes.add_patch(zoom_box)

plt.show()

In [ ]:
abi_x_lowres, abi_y_lowres = np.meshgrid(abi_x[::4], abi_y[::4])
abi_x_hires, abi_y_hires = np.meshgrid(abi_x, abi_y)

In [ ]:
abi_values = np.array(abi_rad[::4,::4]).flatten()

dims = (abi_values.shape[0], 2)
abi_points =  np.zeros(dims)

abi_points[:, 0] = abi_x_lowres.flatten()
abi_points[:, 1] = abi_y_lowres.flatten()

In [ ]:
abi_rad_hires_nn = scipy.interpolate.griddata(abi_points, abi_values, (abi_x_hires, abi_y_hires), method="nearest")
abi_rad_hires_lin = scipy.interpolate.griddata(abi_points, abi_values, (abi_x_hires, abi_y_hires), method="linear")
abi_rad_hires_cube = scipy.interpolate.griddata(abi_points, abi_values, (abi_x_hires, abi_y_hires), method="cubic")

In [ ]:
gridOut = [[abi_rad[::4,::4], abi_rad_hires_nn], [abi_rad_hires_lin, abi_rad_hires_cube]]
labels = [["Re-binning", "Nearest Neighbor"],["Linear", "Cubic"]]
plt.rcParams.update({"font.size": 18, "figure.figsize": [20, 20]})

In [ ]:
fig, axes = plt.subplots(ncols=2, nrows=2, figsize=[8, 8], \
     subplot_kw={"projection": geo_crs})

for i, axis in enumerate(axes):
    for j, ax in enumerate(axis):
        ax.set_extent(extent_jm, crs=ccrs.PlateCarree())
        ax.coastlines("10m", color="black")
        ax.set_title(labels[i][j])
        compare = ax.imshow(gridOut[i][j], cmap="tab20c_r", origin="upper", \
            extent=abi_extent, vmin=abi_lims[0], vmax=abi_lims[1], \
            transform=geo_crs)

fig.colorbar(compare, orientation="horizontal", ax=axes, pad=0.1, 
             label=abi_label, shrink=0.8)

plt.show()

### 8.3.3 Resampling

In [ ]:
filename = "JRR-AOD_v3r2_j01_s202306071807538_e202306071809183_c202306071842528.nc"
aod_file = Path("./data/aod") / filename
aod_nc = xr.open_dataset(aod_file)

In [ ]:
aod = aod_nc["AOD550"]
aod_lat = aod_nc["Latitude"]
aod_lon = aod_nc["Longitude"]

In [ ]:
# Input list of swath points
aod_latlon_hires = geometry.SwathDefinition(lons=aod_lon, lats=aod_lat)

In [ ]:
# Create a new grid at 0.1 degree gridding
x = np.arange(aod_lon.min(), aod_lon.max(), 0.1)
y = np.arange(aod_lat.min(), aod_lat.max(), 0.1)
aod_lon_lowres, aod_lat_lowres = np.meshgrid(x, y)

In [ ]:
# define the new grid using 
aod_latlon_lowres = geometry.GridDefinition(lons=aod_lon_lowres, lats=aod_lat_lowres)

In [ ]:
# Resample the data
aod_lowres = resample_nearest(aod_latlon_hires, np.array(aod), \
    aod_latlon_lowres, radius_of_influence=5000, fill_value=None)

In [ ]:
lims = [0,5]
fig, axes = plt.subplots(nrows=2,  figsize=[15, 10], \
     subplot_kw={"projection": ccrs.PlateCarree()})

for ax in axes:
    ax.coastlines("10m", color="black")

axes[0].set_title("Before regridding")
top = axes[0].pcolormesh(aod_lon, aod_lat, aod, \
        vmin=lims[0], vmax=lims[1], transform=ccrs.PlateCarree())

axes[1].set_title("After regridding")
bottom = axes[1].pcolormesh(aod_lon_lowres, aod_lat_lowres, aod_lowres, \
        vmin=lims[0], vmax=lims[1], transform=ccrs.PlateCarree())

fig.colorbar(bottom, orientation="horizontal", ax=axes, pad=0.1, 
             label="Aerosol Optical Depth [unitless]", shrink=0.8)

plt.show()